# Assignment 2: Physics-Informed Neural Network for the Wave Equation (100 points)

In this assignment, you will extend the PINN framework to solve the 1D wave equation — a different class of PDE that requires handling **second-order time derivatives**.

## Background

The 1D wave equation describes the propagation of waves:

$$u_{tt} = c^2 u_{xx}$$

where $c > 0$ is the wave speed.

**Domain:** $t \in [0, 1]$, $x \in [0, 1]$

**Initial conditions:**
- $u(0, x) = \sin(\pi x)$ (initial displacement)
- $u_t(0, x) = 0$ (initially at rest)

**Boundary conditions:** $u(t, 0) = 0$, $u(t, 1) = 0$ (Dirichlet)

**Wave speed:** $c = 1$

**Analytical solution:** $u(t, x) = \cos(c\pi t)\sin(\pi x)$

**Key difference from the heat equation:** The wave equation has a **second-order time derivative** $u_{tt}$, requiring an additional IC for $u_t(0, x)$ and an extra level of automatic differentiation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas. Do not rename or delete any provided functions or classes.

---

## Part 1: Verify the Analytical Solution (12 points)

**[Non-coding]** Prove that $u(t, x) = \cos(c\pi t)\sin(\pi x)$ with $c = 1$ satisfies:

1. (4 points) The PDE: $u_{tt} - c^2 u_{xx} = 0$. Compute $u_{tt}$ and $u_{xx}$ and substitute.
2. (3 points) The initial displacement: $u(0, x) = \sin(\pi x)$.
3. (3 points) The initial velocity: $u_t(0, x) = 0$. You must compute $u_t$ first.
4. (2 points) The boundary conditions: $u(t, 0) = 0$ and $u(t, 1) = 0$.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Build the WavePINN Model (10 points)

**[Coding]** Implement `WavePINN` with the same architecture as `HeatPINN`.

**Specification:**
- Input: $(B, 2)$ — columns are $t$ and $x$
- Architecture: 4 hidden layers of 64 units with Tanh activations
- Output: $(B, 1)$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class WavePINN(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, tx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: Wave Equation PDE Residual (15 points)

**[Coding]** Implement `compute_wave_residual` that computes $u_{tt} - c^2 u_{xx}$.

**This is harder than the heat equation** because you need **two** second-order derivatives ($u_{tt}$ and $u_{xx}$), each requiring two calls to `autograd.grad`.

**Specification:**
- Compute first-order derivatives: $u_t$ and $u_x$ from $u$ w.r.t. $tx$
- Compute $u_{tt}$ from $u_t$ w.r.t. $tx$ (take the $t$-component)
- Compute $u_{xx}$ from $u_x$ w.r.t. $tx$ (take the $x$-component)
- Return $u_{tt} - c^2 u_{xx}$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_wave_residual(model, tx, c=1.0):
    """
    Compute the wave equation residual: u_tt - c^2 * u_xx
    
    Args:
        model: WavePINN instance
        tx: tensor of shape (B, 2)
        c: wave speed
    
    Returns:
        residual: tensor of shape (B, 1)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Initial Velocity Condition (10 points)

**[Coding]** The wave equation has TWO initial conditions: $u(0, x) = \sin(\pi x)$ and $u_t(0, x) = 0$.

Implement a function `compute_initial_velocity` that computes $u_t$ at $t = 0$ for given $x$ values.

**Specification:**
- Input: `model`, `tx_ic` of shape $(N, 2)$ where all $t = 0$
- Use `autograd.grad` to compute $\frac{\partial u}{\partial t}$ at these points
- Return $u_t$ of shape $(N, 1)$

Then create the IC dataset with BOTH conditions:
- `self.tx`: shape $(N, 2)$ with $t = 0$
- `self.u`: shape $(N, 1)$ with $\sin(\pi x)$ (displacement IC)
- `self.u_t`: shape $(N, 1)$ with zeros (velocity IC)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_initial_velocity(model, tx_ic):
    """
    Compute u_t at the initial condition points.
    
    Args:
        model: WavePINN instance
        tx_ic: tensor of shape (N, 2) with t=0
    
    Returns:
        u_t: tensor of shape (N, 1)
    """
    pass  # YOUR CODE


class WaveICDataset(Dataset):
    def __init__(self, n_points=100):
        pass  # YOUR CODE

    def __len__(self):
        pass  # YOUR CODE

    def __getitem__(self, idx):
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 5: Create All Datasets (8 points)

**[Coding]** Create:
- PDE dataset: 10,000 random collocation points with a DataLoader (batch_size=256, shuffle=True)
- IC dataset: 100 points (use `WaveICDataset`)
- BC dataset: 100 time points at each boundary ($x = 0$ and $x = 1$), both with $u = 0$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Create all datasets and the PDE DataLoader

""" END OF THIS PART """

## Part 6: Four-Term Loss Function (10 points)

**[Non-coding + Coding]** The wave equation PINN has **four** loss terms:

$$\mathcal{L} = \mathcal{L}_{PDE} + \mathcal{L}_{IC,u} + \mathcal{L}_{IC,u_t} + \mathcal{L}_{BC}$$

1. (4 points) **[Non-coding]** Explain why the wave equation requires a fourth loss term ($\mathcal{L}_{IC,u_t}$) that the heat equation does not need. What is the physical meaning of this constraint?

2. (6 points) **[Coding]** Implement a function `compute_total_loss` that computes all four loss terms and returns their sum. The function should also return the individual loss values for logging.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_total_loss(model, tx_pde, tx_ic, u_ic, u_t_ic, tx_bc, u_bc, c=1.0):
    """
    Compute the total PINN loss for the wave equation.
    
    Returns:
        total_loss: scalar
        loss_dict: dict with keys 'pde', 'ic_u', 'ic_ut', 'bc'
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 7: Training Loop (10 points)

**[Coding]** Train the WavePINN.

- 3000 epochs with Adam, lr=1e-3
- Mini-batch PDE data, full IC and BC data
- Print loss every 500 epochs (total + individual components)
- Store loss history

In [ ]:
### WRITE YOUR SOLUTION HERE ###

model = WavePINN(hidden_dim=64, num_layers=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_history = []

# Training loop

""" END OF THIS PART """

## Part 8: Compare Heat and Wave Solutions (7 points)

**[Non-coding]** Answer:

1. (3 points) The heat equation solution $e^{-\alpha\pi^2 t}\sin(\pi x)$ decays exponentially. The wave equation solution $\cos(c\pi t)\sin(\pi x)$ oscillates. How does this difference affect the difficulty of the PINN training?

2. (2 points) Which PDE do you expect to have larger training error, and why?

3. (2 points) Would increasing the number of collocation points help more for the heat equation or the wave equation? Explain.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 9: Evaluation and Visualization (10 points)

**[Coding]** Evaluate the trained WavePINN.

1. (4 points) Create a 50×50 test grid. Compute PINN predictions and the analytical solution.
2. (3 points) Print max error, mean error, and relative L2 error.
3. (3 points) Plot: (a) PINN prediction, (b) Analytical solution, (c) Absolute error — all as heatmaps.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Evaluation and visualization

""" END OF THIS PART """

## Part 10: Extension — Higher Wave Modes (8 points)

**[Coding]** The solution $u(t, x) = \cos(n c \pi t)\sin(n\pi x)$ is valid for any positive integer $n$ (the mode number). Higher modes oscillate faster.

1. (3 points) Verify (by substitution) that $u(t, x) = \cos(2c\pi t)\sin(2\pi x)$ satisfies the wave equation with the same $c$.

2. (5 points) Train a new WavePINN for the $n = 2$ mode: IC is $u(0, x) = \sin(2\pi x)$, $u_t(0, x) = 0$, same BC. Use the same architecture. Report the final error and compare to $n = 1$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# n=2 mode training

""" END OF THIS PART """